# Export PocketTTS models to ONNX + upload to HF

Produces the four ONNX files `PocketTTSEngine` on the Android side expects, uploads them to `HereLiesAz/liperty-pocket-tts`. `setup_libs.sh` pulls from there into `app/src/main/assets/`.

This replaces the GitHub-Release-based download path in the previous version of `setup_libs.sh`'s TTS block, which is dead (the release tag `v0.1.0-models` doesn't host the actual .onnx files).


## 1. Setup


In [6]:
import os, sys
import torch

IS_KAGGLE = os.path.exists("/kaggle/working") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ENV = "kaggle" if IS_KAGGLE else "local"
print(f"Environment: {ENV}")
print(f"Python: {sys.version.split()[0]}, PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")


Environment: kaggle
Python: 3.12.12, PyTorch: 2.10.0+cu128
CUDA: True


In [7]:
%%capture
# Pin versions that are known to work for VITS ONNX export. Coqui TTS
# >= 0.22 introduced an `inference` API that's easier to wrap; older
# versions need to monkeypatch the forward path. SpeechBrain >= 0.5
# is required for the EncoderClassifier ONNX-trace friendliness.
!pip install -q \
    "huggingface_hub>=0.27,<1.0" \
    "speechbrain>=0.5.16,<1.1" \
    "TTS==0.22.0" \
    "onnx>=1.16" \
    "onnxruntime>=1.18" \
    "onnxscript" \
    "numpy" \
    "scipy"
print("Deps installed.")


In [8]:
WORK_DIR = "/kaggle/working/work" if IS_KAGGLE else "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)
print(f"Work dir: {WORK_DIR}")


Work dir: /kaggle/working/work


In [9]:
from huggingface_hub import login, whoami

token = os.environ.get("HF_TOKEN")
if not token and IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if token:
    login(token, add_to_git_credential=True)
else:
    from huggingface_hub import notebook_login
    notebook_login()
print(f"HF user: {whoami()['name']}")


Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
HF user: HereLiesAz


## 2. Speaker encoder (SpeechBrain ECAPA-TDNN)

Input: 16 kHz mono audio. Output: 192-d speaker embedding. ~5-7 MB.


In [11]:
import torch
import torch.nn as nn

SPEAKER_ONNX = os.path.join(WORK_DIR, "pocket_tts_speaker.onnx")

if not os.path.exists(SPEAKER_ONNX):
    from speechbrain.inference.speaker import EncoderClassifier
    print("Loading SpeechBrain ECAPA-TDNN ...")
    classifier = EncoderClassifier.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir=os.path.join(WORK_DIR, "spkrec-ecapa-voxceleb"),
    )

    class SpeakerEncoderWrapper(nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, audio):
            # audio: (1, T_samples) at 16 kHz, peak-normalized
            emb = self.m.encode_batch(audio)
            return emb.squeeze(1)   # (1, 192)

    wrapper = SpeakerEncoderWrapper(classifier).eval()
    dummy = torch.randn(1, 16000 * 3)

    print("Exporting ...")
    torch.onnx.export(
        wrapper, dummy, SPEAKER_ONNX,
        input_names=["audio"],
        output_names=["embedding"],
        dynamic_axes={"audio": {1: "audio_length"}},
        opset_version=14,
    )

import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(SPEAKER_ONNX, providers=["CPUExecutionProvider"])
test_audio = np.random.randn(1, 48000).astype(np.float32)
emb = sess.run(None, {"audio": test_audio})[0]
print(f"Speaker encoder OK. Output shape: {emb.shape} (expected (1, 192))")
print(f"Size: {os.path.getsize(SPEAKER_ONNX) / 1e6:.1f} MB")


ModuleNotFoundError: No module named 'speechbrain'

## 3. VITS acoustic model (Coqui VCTK multispeaker)

VITS is end-to-end: phoneme IDs + speaker embedding -> waveform at 22050 Hz. ~35 MB.


In [12]:
ACOUSTIC_ONNX = os.path.join(WORK_DIR, "pocket_tts_acoustic.onnx")

if not os.path.exists(ACOUSTIC_ONNX):
    from TTS.api import TTS

    print("Loading Coqui VITS VCTK ...")
    tts = TTS(model_name="tts_models/en/vctk/vits", progress_bar=False)
    vits = tts.synthesizer.tts_model
    vits.eval()

    class VITSAcousticWrapper(nn.Module):
        """Wraps VITS.inference() in a tensor-in/tensor-out shape ONNX can
        trace. VITS internally handles attention masks, mel synthesis,
        and HiFi-GAN vocoding — what comes out is a 22050 Hz waveform."""

        def __init__(self, vits): super().__init__(); self.vits = vits

        def forward(self, input_ids, speaker_ids):
            # input_ids: (1, T_text) long phoneme indices
            # speaker_ids: (1, 192) float speaker embedding (d_vector)
            x_lengths = torch.tensor([input_ids.shape[1]], dtype=torch.long)
            outputs = self.vits.inference(
                input_ids, x_lengths, sid=None, d_vectors=speaker_ids,
            )
            return outputs["wav"]   # (1, T_audio)

    wrapper = VITSAcousticWrapper(vits)
    dummy_ids = torch.randint(0, 40, (1, 12))
    dummy_speaker = torch.randn(1, 192)

    print("Exporting (this can take a minute) ...")
    torch.onnx.export(
        wrapper, (dummy_ids, dummy_speaker), ACOUSTIC_ONNX,
        input_names=["input_ids", "speaker_ids"],
        output_names=["waveform"],
        dynamic_axes={
            "input_ids": {1: "text_length"},
            "waveform":  {1: "audio_length"},
        },
        opset_version=14,
    )

sess = ort.InferenceSession(ACOUSTIC_ONNX, providers=["CPUExecutionProvider"])
print(f"Acoustic inputs: {[i.name for i in sess.get_inputs()]}")
print(f"Acoustic outputs: {[o.name for o in sess.get_outputs()]}")
print(f"Size: {os.path.getsize(ACOUSTIC_ONNX) / 1e6:.1f} MB")


ModuleNotFoundError: No module named 'TTS'

## 4. Pass-through vocoder

VITS produces waveform directly, but `PocketTTSEngine.generateAudio()` is hardcoded for a 2-stage pipeline (acoustic -> vocoder). To preserve that contract without rewriting the engine, we ship an identity-function ONNX in the vocoder slot.

The engine sets `vocoderInputName` (default `"mel"`) on the input dict, so this ONNX must use input name `mel`. Output is identical to input.


In [ ]:
VOCODER_ONNX = os.path.join(WORK_DIR, "pocket_tts_vocoder.onnx")

if not os.path.exists(VOCODER_ONNX):
    class PassthroughVocoder(nn.Module):
        def forward(self, mel):
            return mel

    wrapper = PassthroughVocoder()
    dummy = torch.randn(1, 22050)
    torch.onnx.export(
        wrapper, dummy, VOCODER_ONNX,
        input_names=["mel"],
        output_names=["waveform"],
        dynamic_axes={"mel": {1: "t"}, "waveform": {1: "t"}},
        opset_version=14,
    )

sess = ort.InferenceSession(VOCODER_ONNX, providers=["CPUExecutionProvider"])
test = np.random.randn(1, 1000).astype(np.float32)
out = sess.run(None, {"mel": test})[0]
print(f"Vocoder OK (pass-through). In={test.shape} Out={out.shape}")
print(f"Size: {os.path.getsize(VOCODER_ONNX) / 1e6:.2f} MB")


## 5. Phoneme map for the Android side

The Coqui VITS VCTK model expects phoneme indices defined by its OWN tokenizer (espeak-ng-derived IPA), not Liperty's MLConstants.PHONEME_VOCAB (which is 40-symbol ARPABET).

PocketTTSEngine.kt currently does the wrong thing: it tokenizes via ARPABET and passes those indices to the VITS acoustic ONNX. The model produces garbage because the index space doesn't match.

This cell dumps the VITS character-to-index map alongside the ONNX so the Android side can use it to tokenize correctly. The follow-up fix on the engine side is to read this map at init and route Liperty's text -> espeak phonemes -> VITS indices instead of -> ARPABET -> VITS indices.


In [ ]:
import json as _json

PHONEME_MAP_PATH = os.path.join(WORK_DIR, "pocket_tts_phoneme_map.json")

if not os.path.exists(PHONEME_MAP_PATH):
    from TTS.api import TTS
    tts = TTS(model_name="tts_models/en/vctk/vits", progress_bar=False)
    vits = tts.synthesizer.tts_model
    # The tokenizer is on the config; characters list defines the
    # index ordering. eSpeak phonemizer handles graphemes -> phonemes
    # at synthesis time on Python; we'd need to port that or use the
    # character-level input.
    tokenizer = tts.synthesizer.tts_config.characters
    pad = tts.synthesizer.tts_config.characters.pad or ""
    eos = tts.synthesizer.tts_config.characters.eos or ""
    bos = tts.synthesizer.tts_config.characters.bos or ""
    blank = tts.synthesizer.tts_config.characters.blank or ""
    chars = list(tts.synthesizer.tts_config.characters.characters)
    phonemes = list(tts.synthesizer.tts_config.characters.phonemes or "")

    pmap = {
        "characters": chars,
        "phonemes": phonemes,
        "pad": pad, "eos": eos, "bos": bos, "blank": blank,
        "is_phoneme": True if phonemes else False,
        "sample_rate": int(tts.synthesizer.output_sample_rate),
    }
    with open(PHONEME_MAP_PATH, "w") as f:
        _json.dump(pmap, f, indent=2)

with open(PHONEME_MAP_PATH) as f:
    pmap = _json.load(f)
print(f"Phoneme map sample_rate: {pmap['sample_rate']} Hz")
print(f"Characters ({len(pmap['characters'])}): {pmap['characters'][:20]}...")
print(f"Phonemes ({len(pmap['phonemes'])}): {pmap['phonemes'][:20]}...")


## 6. Upload to HF


In [ ]:
from huggingface_hub import HfApi, create_repo

REPO = "HereLiesAz/liperty-pocket-tts"
create_repo(REPO, repo_type="model", private=False, exist_ok=True)
api = HfApi()
for path in (SPEAKER_ONNX, ACOUSTIC_ONNX, VOCODER_ONNX, PHONEME_MAP_PATH):
    if not os.path.exists(path): continue
    api.upload_file(
        path_or_fileobj=path,
        path_in_repo=os.path.basename(path),
        repo_id=REPO, repo_type="model",
        commit_message=f"PocketTTS {os.path.basename(path)} ({os.path.getsize(path) // 1024} KB)",
    )
    print(f"Uploaded {os.path.basename(path)}")
print()
print(f"All assets at: https://huggingface.co/{REPO}")
print()
print("Next steps on the Android side:")
print("  1. setup_libs.sh: change TTS block to pull from this repo")
print("  2. PocketTTSEngine.kt: use pocket_tts_phoneme_map.json instead of")
print("     MLConstants.PHONEME_VOCAB for tokenization")
print("  3. PocketTTSEngine.generateAudio: vocoder is now a pass-through,")
print("     skip it (or keep it as a no-op pipeline node)")
print("  4. AudioRouter: VITS outputs 22050 Hz, NOT 16 kHz — set the")
print("     AudioTrack sample rate accordingly")
